# Exploratory Analysis — Branch Traces & Predictability

This notebook explores the synthetic branch trace and motivates the feature
design used by the predictors. Run it after `pip install -e .` (or `make setup`).

It covers:
1. Overall taken-rate and per-PC bias.
2. How each branch *pattern* behaves.
3. How prediction accuracy scales with global-history length — the empirical
   justification for using history at all.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from branchpred.data.trace_generator import generate_trace
from branchpred.baselines.two_bit_predictor import TwoBitPredictor
from branchpred.baselines.gshare_predictor import GsharePredictor
from branchpred.evaluation.evaluator import evaluate_online

df = generate_trace(num_branches=100_000, seed=42)
print(f"branches={len(df):,}  distinct PCs={df['pc'].nunique()}  taken-rate={df['outcome'].mean():.3f}")
df.head()

## 1. Per-PC bias

Real programs have a mix of strongly-biased branches (loops) and hard-to-predict
ones. The distribution of per-PC taken-rates shows both extremes.

In [ ]:
per_pc = df.groupby('pc')['outcome'].mean()
plt.figure(figsize=(7, 4))
plt.hist(per_pc.values, bins=20, color='#4C72B0', edgecolor='white')
plt.xlabel('per-PC taken rate'); plt.ylabel('number of branch sites')
plt.title('Distribution of per-PC bias'); plt.grid(alpha=0.3); plt.show()

## 2. Behaviour per pattern

Generate single-pattern traces to see each modelled behaviour in isolation.

In [ ]:
for pat in ['loop', 'nested_loop', 'correlated', 'function', 'random']:
    d = generate_trace(20_000, pattern_mix={pat: 1.0}, seed=1)
    print(f"{pat:12s} taken-rate={d['outcome'].mean():.3f}")

## 3. Accuracy vs global-history length

gshare's accuracy on a history-correlated workload should improve as it is given
more global-history bits — the empirical reason ML predictors use history.

In [ ]:
corr = generate_trace(60_000, pattern_mix={'correlated': 1.0}, seed=9)
hist_lengths = [0, 2, 4, 6, 8, 12]
accs = []
for h in hist_lengths:
    p = GsharePredictor(table_bits=14, history_bits=h)
    res = evaluate_online(p, corr, score_from=len(corr) // 2)
    accs.append(res.metrics.accuracy)

plt.figure(figsize=(7, 4))
plt.plot(hist_lengths, accs, 'o-', color='#55A868')
plt.xlabel('gshare global-history bits'); plt.ylabel('accuracy')
plt.title('More history helps on correlated branches'); plt.grid(alpha=0.3); plt.show()
print(dict(zip(hist_lengths, [round(a, 3) for a in accs])))

**Takeaway:** with zero history bits gshare degenerates toward the bimodal
predictor and cannot capture correlation; accuracy climbs as history is added.
The perceptron generalises this idea by *learning per-PC weights* over the
history bits. See `scripts/run_comparison.py` for the full head-to-head.